# Compare All Tensile Tests

Overlays the engineering stress-strain curves of every MTS test and tabulates UTS and elongation. The loading and stress-strain steps live in `tensile.py`, next to this notebook.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from tensile import list_test_files, load_test, stress_strain, test_label

# Paths relative to this notebook: they work on any computer
data_dir = os.path.join("..", "data", "MTS_metals")
results_dir = os.path.join("..", "results")

# The corrupted file is cleaned in MASE6401_Adv_Pandas; leave it out here
test_files = [f for f in list_test_files(data_dir) if "Corrupted" not in f]
test_files

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
rows = []

for name in test_files:
    metadata, data = load_test(os.path.join(data_dir, name))
    curve = stress_strain(data, metadata)
    ax.plot(curve["Strain"], curve["Stress (MPa)"], label=test_label(name))
    rows.append({
        "Test": test_label(name),
        "UTS (MPa)": curve["Stress (MPa)"].max(),
        "Elongation (%)": 100 * curve["Strain"].max(),
    })

ax.set_xlabel("Engineering strain (-)")
ax.set_ylabel("Engineering stress (MPa)")
ax.legend()
fig.tight_layout()

os.makedirs(results_dir, exist_ok=True)
fig.savefig(os.path.join(results_dir, "all_tests.png"), dpi=150)

In [ ]:
summary = pd.DataFrame(rows).round(1)
summary